# 最終資料品質驗收與 Backtest Data Contract（唯讀、離線）

依序跑第 1–6 格，約 5–10 分鐘。全程不發網路請求、不呼叫富邦、不改寫任何 raw、帳本或先前的輸出。

輸出寫到新的版本化資料夾：

| 檔案 | 內容 |
|---|---|
| `quality_mask_symbol_day.parquet` | 每個股票日的品質遮罩 |
| `quarantine_exchange_dates.csv`、`quarantine_symbol_days.csv` | 隔離的交易所日期與股票日 |
| `anomaly_ledger.csv` | 其餘未解決缺棒的逐段清單 |
| `acceptance_rule_v2.json` | 規則 v2 的前提、證據與決定 |
| `stats_*.csv` | 各項統計 |
| `QUALITY_REPORT.md` | 報告 |
| `quality_manifest.json` | 所有輸出的 SHA-256 |
| `BACKTEST_DATA_CONTRACT.md`、`backtest_data_contract.json` | 回測資料使用契約 |

規格見 `docs/BACKTEST_DATA_CONTRACT.md` 與 `docs/AUDIT.md` 第 15 節。

In [ ]:
# 1. 參數（皆為先前的不可變輸出）
REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
AUD = f"{DATA_ROOT}/_tw-market-data-lake/audits"
LEDGER_DIR = f"{AUD}/gap_ledger_20261026_163358_3e8eed0"
TC = f"{AUD}/targeted_checks"
ELIG_DIR = f"{TC}/20261030_010831_63ffd6f/eligibility_20261030_010542_63ffd6f"          # 已帶入價格比對
PRICE_DIR = f"{TC}/20261030_003807_4308c60/price_check_20261029_235425_4308c60"
REFETCH_CSV = f"{TC}/20261029_233527_4308c60/official_refetch_20261029_233355_4308c60/official_refetch_compare.csv"
OVER_DIR = f"{TC}/20261030_003807_4308c60/overcount_diag_20261029_235407_4308c60"
FUBON_DIR = f"{TC}/20261030_003807_4308c60/overcount_fubon_20261030_003259_4308c60"
OUT_DIR = "/content/tw_lake_quality"
PROJECT_DIR = f"/content/{REPO_NAME}"

In [ ]:
# 2. 掛載 Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. clone / pull + 依賴 + 依賴
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip())
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)
def stream(args):
    env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def ensure_drive():
    """Mount Drive safely. If an earlier cell unmounted Drive and then wrote under /content/drive,
    those files are only on this runtime's disk: move them aside (never delete) and copy any audit
    folders that are missing on the real Drive."""
    import os, shutil, time
    from google.colab import drive
    if os.path.ismount("/content/drive"):
        return
    stray = None
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        stray = f"/content/drive_local_{int(time.time())}"
        shutil.move("/content/drive", stray)
        print("moved local (unsynced) files to", stray)
    drive.mount("/content/drive")
    if stray:
        src_root = f"{stray}/MyDrive/00Quant_Research/data/_tw-market-data-lake/audits"
        dst_root = f"{DATA_ROOT}/_tw-market-data-lake/audits"
        for dirpath, dirnames, filenames in os.walk(src_root):
            rel = os.path.relpath(dirpath, src_root)
            for d in list(dirnames):
                target = os.path.join(dst_root, rel, d)
                if not os.path.exists(target):
                    shutil.copytree(os.path.join(dirpath, d), target)
                    print("copied missing", target)
                    dirnames.remove(d)
print("Python", sys.version.split()[0])

In [ ]:
# 4. Preflight
for p in (LEDGER_DIR, f"{ELIG_DIR}/symbol_day_quality.parquet", f"{PRICE_DIR}/price_check_symbol_day.parquet",
          REFETCH_CSV, f"{OVER_DIR}/overcount_symbol_days.parquet", f"{FUBON_DIR}/crosscheck.parquet",
          f"{LEDGER_DIR}/gap_runs"):
    print("OK " if os.path.exists(p) else "MISSING ", p)

In [ ]:
# 5. 執行（唯讀、離線）
import glob
print("exit", stream([sys.executable, "-m", "tw_lake.audit.quality_mask", "--ledger-dir", LEDGER_DIR,
                      "--eligibility-dir", ELIG_DIR, "--price-check-dir", PRICE_DIR, "--refetch-evidence", REFETCH_CSV,
                      "--overcount-dir", OVER_DIR, "--fubon-overcount-dir", FUBON_DIR, "--out-dir", OUT_DIR]))
QDIR = sorted(glob.glob(f"{OUT_DIR}/quality_mask_*"))[-1]

In [ ]:
# 6. 複製到 Drive（新資料夾，不覆寫）
import shutil
dest = f"{AUD}/quality_acceptance/{os.path.basename(QDIR)}"
shutil.copytree(QDIR, dest)
print("copied to", dest)
from google.colab import drive
drive.flush_and_unmount()